In [ ]:
!pip install timm yacs wandb gdown

In [ ]:
!git clone https://github.com/Visoura/guided_attention_experiments.git
%cd guided_attention_experiments/transreid_pytorch

In [ ]:
!mkdir -p ../data
!gdown 1VK6OT-IAiXsOq36mkXiL2Ri4eofB8JWP -O ../data/Market-1501_With_Bags.zip
!unzip -q ../data/Market-1501_With_Bags.zip -d ../data/
!rm ../data/Market-1501_With_Bags.zip


In [ ]:
!mkdir -p ../weights
# 1-iko_cKpCjwr7zBkwone9y6LFmSOqmfk
!gdown 1-iko_cKpCjwr7zBkwone9y6LFmSOqmfk -O ../weights/checkpoint0260.pth

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
secret = "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"
try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except:
    wandb.login()

In [ ]:
RUN_NAME = "transreid_guided_attn_vits_koleo_guidedattention_v1"
CONFIG_FILE = "configs/market/vit_small.yml" 
OUTPUT_DIR = "./logs/" + RUN_NAME
LR = 0.0004
BATCH_SIZE = 64
MASK_THRESHOLD = 0.1
SCALE_RATIO = 0.8
PRETRAIN_PATH = "../weights/checkpoint0260.pth"
HW_RATIO = 2
kl_weight = 0.8

!python train.py --config_file {CONFIG_FILE} \
    OUTPUT_DIR {OUTPUT_DIR} \
    DATASETS.ROOT_DIR "../data" \
    MODEL.PRETRAIN_CHOICE 'imagenet' \
    MODEL.PRETRAIN_PATH {PRETRAIN_PATH} \
    MODEL.PRETRAIN_HW_RATIO {HW_RATIO} \
    MODEL.USE_KOLEO_LOSS True \
    MODEL.KOLEO_LOSS_WEIGHT {kl_weight} \
    SOLVER.BASE_LR {LR} \
    SOLVER.IMS_PER_BATCH {BATCH_SIZE} \
    SOLVER.EVAL_PERIOD 20 \
    SOLVER.MAX_EPOCHS 160 \
    MODEL.GUIDED_ATTENTION_TRAIN True \
    MODEL.GUIDED_ATTENTION_TEST True \
    MODEL.GUIDED_SCALE_RATIO {SCALE_RATIO} \
    MODEL.MASK_THRESHOLD {MASK_THRESHOLD} \
    WANDB.ENABLED True \
    WANDB.PROJECT_NAME "PersonViT" \
    WANDB.RUN_NAME "{RUN_NAME}"